# 2026-09-20：WT 诱导表达与突变株未检出验证

输入为 `20260920detail.csv`，包含 Tub、AoGPR200、AoGPRPR8。用户确认 Cp1–Cp3 是同一 cDNA 的三个技术重复；每条件目前一个 cDNA，无独立生物学重复。

输出两张独立论文用图：① WT baseline 与 12hpe 的 Tub 归一化相对表达；② WT 与对应突变株在两个条件下的检出率。零 Cp 按实验约定视为未检出，表达量不赋值为零；未测的交叉突变株–基因组合不补值。样本名前缀 `_` 表示删除突变株 `Δ`，baseline 不额外推断为某个具体小时。

表达分析使用 $2^{-\Delta\Delta Ct}$，目标和 Tub 的技术重复先分别在 Cp 空间平均。假设目标与内参的扩增效率相近且接近 100%；没有效率或扩增曲线数据来单独验证该假设。检出率按照文件中的 Cp 编码统计，不额外推断检出限。

[比较 Ct 方法](https://apps.thermofisher.com/apps/help/MAN0010505/GUID-FFD3B71A-7791-4714-897F-EDE325F745EA.html)。

## Manuscript integration copy (20 September 2026)
This portable copy reads the bundled input snapshot and writes to `reproduced/`. The author confirmed exposure to live C. elegans and a two-step cDNA/qPCR workflow. Input data and calculations are unchanged. Manuscript Figure S1 = analysis Figure1; Figure S2 = analysis Figure2.

In [1]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
from IPython.display import display, HTML

PROJECT_DIR = Path.cwd()
SOURCE = PROJECT_DIR / '20260920detail.input.csv'
OUT = PROJECT_DIR / 'reproduced'
OUT.mkdir(parents=True, exist_ok=True)
SNAPSHOT = OUT / '20260920detail.input.csv'
# 首次保存原始输入；之后使用快照保证复现。更换数据时请另存新的分析目录。
if not SNAPSHOT.exists():
    SNAPSHOT.write_bytes(SOURCE.read_bytes())
raw = pd.read_csv(SNAPSHOT, encoding='utf-8-sig')
required = ['cDNA', 'probe_name', 'well_rep1', 'well_rep2', 'well_rep3', 'Cp1', 'Cp2', 'Cp3']
assert set(required).issubset(raw.columns)
assert not raw.duplicated(['cDNA', 'probe_name']).any()
assert set(raw.probe_name) == {'Tub', 'AoGPR200', 'AoGPRPR8'}
assert len(raw) == 14
assert not raw[['well_rep1', 'well_rep2', 'well_rep3']].stack().duplicated().any()
cp_cols = ['Cp1', 'Cp2', 'Cp3']
for col in cp_cols:
    raw[col] = pd.to_numeric(raw[col], errors='raise')
assert raw[cp_cols].notna().all().all() and np.isfinite(raw[cp_cols]).all().all()
assert raw[cp_cols].ge(0).all().all()
summary = raw.copy()
summary['sample_name'] = summary.cDNA.str.replace('_', 'Δ', regex=False)
summary['condition'] = np.where(summary.cDNA.str.endswith('-12hpe'), '12hpe', 'Baseline')
summary['genotype'] = summary.sample_name.str.replace('-12hpe', '', regex=False)
summary['n_attempted'] = 3
summary['n_detected'] = summary[cp_cols].gt(0).sum(axis=1)
summary['detection_percent'] = summary.n_detected / summary.n_attempted * 100
valid_cp = summary[cp_cols].where(summary[cp_cols].gt(0))
summary['Cp_mean'] = valid_cp.mean(axis=1)
summary['Cp_sd'] = valid_cp.std(axis=1, ddof=1)
summary['Cp_mean_variance'] = summary.Cp_sd ** 2 / summary.n_detected.replace(0, np.nan)
summary['technical_sd_review'] = summary.Cp_sd > 0.5  # 复核提示，不自动删孔
summary['status'] = np.where(summary.n_detected.eq(0), 'ND', 'Detected')
summary.to_csv(OUT / 'Cp_summary.csv', index=False, encoding='utf-8-sig', na_rep='')
long_rows = []
for _, row in summary.iterrows():
    for rep in [1, 2, 3]:
        long_rows.append({'sample': row.sample_name, 'gene': row.probe_name,
                         'technical_replicate': rep, 'well': row[f'well_rep{rep}'],
                         'input_Cp': row[f'Cp{rep}'], 'detected': row[f'Cp{rep}'] > 0})
pd.DataFrame(long_rows).to_csv(OUT / 'well_source_data.csv', index=False, encoding='utf-8-sig')
(OUT / 'input_sha256.txt').write_text(hashlib.sha256(SNAPSHOT.read_bytes()).hexdigest() + '  ' + SNAPSHOT.name + '\n')
display(HTML(summary[['sample_name', 'probe_name', *cp_cols, 'Cp_mean', 'Cp_sd', 'n_detected']].to_html(
    index=False, na_rep='ND', float_format=lambda x: f'{x:.3f}')))


sample_name,probe_name,Cp1,Cp2,Cp3,Cp_mean,Cp_sd,n_detected
WT,Tub,17.400,17.510,17.370,17.427,0.074,3
WT-12hpe,Tub,16.330,16.710,16.260,16.433,0.242,3
ΔAoGPRPR8,Tub,16.880,16.900,17.800,17.193,0.525,3
ΔAoGPRPR8-12hpe,Tub,17.250,17.020,17.370,17.213,0.178,3
ΔAoGPR200,Tub,16.920,16.390,16.130,16.480,0.403,3
ΔAoGPR200-12hpe,Tub,16.540,17.720,16.930,17.063,0.601,3
WT,AoGPR200,21.430,22.820,21.710,21.987,0.735,3
WT-12hpe,AoGPR200,19.130,19.400,19.320,19.283,0.139,3
ΔAoGPR200,AoGPR200,0.000,0.000,0.000,ND,ND,0
ΔAoGPR200-12hpe,AoGPR200,0.000,0.000,0.000,ND,ND,0


## ΔΔCt 与技术不确定性

`ΔCt = mean(Cp_target) − mean(Cp_Tub)`；`ΔΔCt = ΔCt_sample − ΔCt_baseline_WT`；`fold = 2^(−ΔΔCt)`。

每个基因的 WT baseline 是自身校准组，倍数为 1。误差线采用均值方差传播：`Var(ΔCt) = SD_target²/n_target + SD_Tub²/n_Tub`；独立反应的样本 ΔCt 方差与校准 ΔCt 方差相加，得到 ΔΔCt 的技术 SE，随后转换为不对称倍数误差线。校准组自比较误差为 0，这是归一化定义，不代表其原始 Cp 无波动。

图 1 的误差线是 ±1 个传播技术 SE，不是生物学 SD/SEM 或 95% CI；没有 p 值或显著性星号。图 2 为实际检出孔比例，没有把 0/3 检出换算成“表达量为零”。

In [2]:
reference = summary.loc[summary.probe_name.eq('Tub'), ['cDNA', 'Cp_mean', 'Cp_mean_variance', 'n_detected']].rename(columns={
    'Cp_mean': 'Tub_Cp_mean', 'Cp_mean_variance': 'Tub_mean_variance', 'n_detected': 'Tub_n_detected'})
assert reference.Tub_n_detected.eq(3).all()
results = summary.loc[summary.probe_name.ne('Tub')].merge(reference, on='cDNA', validate='many_to_one', how='left')
assert results.Tub_Cp_mean.notna().all()
results['delta_Ct'] = results.Cp_mean - results.Tub_Cp_mean
results['delta_Ct_variance'] = results.Cp_mean_variance + results.Tub_mean_variance
calibrators = results.loc[results.cDNA.eq('WT'), ['probe_name', 'delta_Ct', 'delta_Ct_variance']].rename(columns={
    'delta_Ct': 'calibrator_delta_Ct', 'delta_Ct_variance': 'calibrator_delta_Ct_variance'})
results = results.merge(calibrators, on='probe_name', validate='many_to_one')
results['delta_delta_Ct'] = results.delta_Ct - results.calibrator_delta_Ct
results['fold_change'] = np.exp2(-results.delta_delta_Ct)
results['technical_SE_ddCt'] = np.sqrt(results.delta_Ct_variance + results.calibrator_delta_Ct_variance)
results.loc[results.cDNA.eq('WT'), 'technical_SE_ddCt'] = 0.0
results['fold_lower_1technicalSE'] = np.exp2(-results.delta_delta_Ct - results.technical_SE_ddCt)
results['fold_upper_1technicalSE'] = np.exp2(-results.delta_delta_Ct + results.technical_SE_ddCt)
assert np.allclose(results.loc[results.cDNA.eq('WT'), 'fold_change'], 1)
assert results.loc[results.status.eq('ND'), 'fold_change'].isna().all()
results.to_csv(OUT / 'ddct_results.csv', index=False, encoding='utf-8-sig', na_rep='')
wt_results = results.loc[results.genotype.eq('WT')].copy()
wt_results.to_csv(OUT / 'Figure1_source_data.csv', index=False, encoding='utf-8-sig')
results[['probe_name', 'sample_name', 'genotype', 'condition', 'n_attempted', 'n_detected', 'detection_percent', 'status', 'Tub_n_detected']].to_csv(
    OUT / 'Figure2_source_data.csv', index=False, encoding='utf-8-sig')
display(HTML(results[['probe_name', 'sample_name', 'delta_Ct', 'delta_delta_Ct', 'fold_change', 'status']].to_html(
    index=False, na_rep='ND', float_format=lambda x: f'{x:.3f}')))


probe_name,sample_name,delta_Ct,delta_delta_Ct,fold_change,status
AoGPR200,WT,4.560,0.000,1.000,Detected
AoGPR200,WT-12hpe,2.850,-1.710,3.272,Detected
AoGPR200,ΔAoGPR200,ND,ND,ND,ND
AoGPR200,ΔAoGPR200-12hpe,ND,ND,ND,ND
AoGPRPR8,WT,5.257,0.000,1.000,Detected
AoGPRPR8,WT-12hpe,3.107,-2.150,4.438,Detected
AoGPRPR8,ΔAoGPRPR8,ND,ND,ND,ND
AoGPRPR8,ΔAoGPRPR8-12hpe,ND,ND,ND,ND


## 图 1：WT 中两种受体基因的 12hpe 表达变化

沿用文章配色：对照 `#BEDEC7`，比较组 `#7B61AA`；柱宽 0.7、黑色轮廓、无图例、45° 加粗横轴标签、经典坐标轴，每面板约 3.5 × 5 inch。

柱高保持 ΔΔCt 估计（技术散点的几何均值），误差线保持 ±1 个传播技术 SE。黑色散点是每个目标孔以同一样本 Tub 平均 Cp 归一化后的技术值，不包含内参及校准值的不确定性，不是独立生物学样本。WT 自身校准柱高固定为 1；其散点仍显示目标孔间波动。

In [3]:
import os
os.environ.setdefault('MPLCONFIGDIR', str(OUT / '.mplconfig'))
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

# 对齐文章 ggplot theme_classic 风格；每个基因面板约 3.5 × 5 inch。
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 12,
                     'axes.labelsize': 10, 'axes.titlesize': 12,
                     'xtick.labelsize': 12, 'ytick.labelsize': 12,
                     'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.fonttype': 'none',
                     'axes.linewidth': 0.8, 'savefig.dpi': 600})
COLORS = {'control': '#BEDEC7', 'comparison': '#7B61AA', 'accent': '#ED5996'}
BAR_WIDTH = 0.7
DOT_OFFSETS = np.array([-0.12, 0.0, 0.12])  # 固定位置，复跑不会改变

def save_figure(fig, name):
    for ext in ['pdf', 'svg', 'png']:
        fig.savefig(OUT / f'{name}.{ext}', dpi=600, facecolor='white')

def classic_axes(ax, labels, ylabel):
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['bottom', 'left']].set_color('black')
    ax.tick_params(axis='both', colors='black', width=0.8)
    ax.set_xticks(range(len(labels)), labels, rotation=45, ha='right', rotation_mode='anchor')
    for label in ax.get_xticklabels(): label.set_fontweight('bold')
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_xlim(-0.65, len(labels)-0.35)
    ax.set_xlabel('')
    ax.grid(False)

genes = ['AoGPR200', 'AoGPRPR8']
conditions = ['Baseline', '12hpe']
# 技术散点：每孔目标 Cp 减同一样本 Tub 平均 Cp，再减 WT baseline 平均 ΔCt。
# 不将不同 assay 的第 i 孔配对；这些点只展示目标 assay 的技术离散度。
point_rows = []
for _, row in wt_results.iterrows():
    for rep in [1, 2, 3]:
        point_delta = row[f'Cp{rep}'] - row.Tub_Cp_mean
        point_rows.append({'gene': row.probe_name, 'sample': row.sample_name,
                          'condition': row.condition, 'technical_replicate': rep,
                          'technical_relative_expression': np.exp2(-(point_delta-row.calibrator_delta_Ct))})
technical_points = pd.DataFrame(point_rows)
technical_points.to_csv(OUT / 'Figure1_technical_points.csv', index=False, encoding='utf-8-sig')

fig1, axes1 = plt.subplots(1, 2, figsize=(7, 5), sharey=True)
ymax = max(wt_results.fold_upper_1technicalSE.max(), technical_points.technical_relative_expression.max()) * 1.15
for panel, (ax, gene) in enumerate(zip(axes1, genes)):
    frame = wt_results.loc[wt_results.probe_name.eq(gene)].set_index('condition').loc[conditions]
    for x, (condition, row) in enumerate(frame.iterrows()):
        color = COLORS['control'] if x == 0 else COLORS['comparison']
        ax.bar(x, row.fold_change, width=BAR_WIDTH, color=color, edgecolor='black', linewidth=0.8, zorder=2)
        if row.technical_SE_ddCt > 0:
            # errorbar width=0.15 in ggplot data units
            low, high = row.fold_lower_1technicalSE, row.fold_upper_1technicalSE
            ax.vlines(x, low, high, color='black', linewidth=0.8, zorder=3)
            ax.hlines([low, high], x-0.075, x+0.075, color='black', linewidth=0.8, zorder=3)
        points = technical_points.loc[(technical_points.gene==gene)&(technical_points.condition==condition), 'technical_relative_expression'].to_numpy()
        ax.scatter(x+DOT_OFFSETS, points, s=16, color='black', alpha=0.7, linewidths=0, zorder=4)
    classic_axes(ax, ['WT', 'WT-12hpe'], 'Relative expression (fold)')
    ax.set_ylim(0, ymax)
    ax.yaxis.set_major_locator(MultipleLocator(1))
    ax.set_title(gene, fontstyle='italic', pad=15)
    ax.text(-0.18, 1.04, chr(65+panel), transform=ax.transAxes, fontweight='bold', fontsize=14)
fig1.subplots_adjust(left=0.12, right=0.96, bottom=0.25, top=0.87, wspace=0.35)
save_figure(fig1, 'Figure1_WT_12hpe_induction')
plt.show()


Matplotlib is building the font cache; this may take a moment.
<ipython-input-3-ab706a83fe59>:66: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 图 2：对应删除突变株的受体转录本未检出

每个面板对应一个受体，按 Baseline 与 12hpe 比较 WT 和该受体的删除突变株。柱高为检出率：WT 3/3 = 100%，突变株 0/3 = 0%；零柱位置标注 ND (0/3)。Tub 在所有样本中 3/3 检出。此图支持“该测定未检出转录本”，不单独证明基因组删除或绝对无转录。

黑色散点分别对应三个技术反应（检出=100，未检出=0）。三个点全部相同，因此 SD=0，无可见误差线。图中 Δ200 和 ΔPR8 分别表示 ΔAoGPR200 和 ΔAoGPRPR8。

In [4]:
fig2, axes2 = plt.subplots(1, 2, figsize=(7, 5), sharey=True)
detection_points = []
for panel, (ax, gene) in enumerate(zip(axes2, genes)):
    mutant_name = 'Δ' + gene
    subset = results.loc[results.probe_name.eq(gene)].set_index(['condition', 'genotype'])
    combos = [('Baseline', 'WT'), ('Baseline', mutant_name), ('12hpe', 'WT'), ('12hpe', mutant_name)]
    short_mutant = 'Δ200' if gene == 'AoGPR200' else 'ΔPR8'
    labels = ['WT', short_mutant, 'WT-12hpe', short_mutant+'-12hpe']
    for x, (condition, genotype) in enumerate(combos):
        row = subset.loc[(condition, genotype)]
        color = COLORS['control'] if genotype=='WT' else COLORS['comparison']
        ax.bar(x, row.detection_percent, width=BAR_WIDTH, color=color, edgecolor='black', linewidth=0.8, zorder=2)
        points = np.array([100.0 if row[f'Cp{rep}'] > 0 else 0.0 for rep in [1, 2, 3]])
        ax.scatter(x+DOT_OFFSETS, points, s=16, color='black', alpha=0.7, linewidths=0, zorder=4, clip_on=False)
        for rep, value in enumerate(points, start=1):
            detection_points.append({'gene':gene, 'condition':condition, 'genotype':genotype,
                                     'technical_replicate':rep, 'detection_indicator_percent':value})
        if row.n_detected == 0:
            ax.text(x, 5, 'ND\n(0/3)', ha='center', va='bottom', fontsize=9)
        else:
            ax.text(x, 103, '3/3', ha='center', va='bottom', fontsize=9)
    classic_axes(ax, labels, 'Target-positive reactions (%)')
    ax.set_ylim(0, 115)
    ax.set_yticks([0, 25, 50, 75, 100])
    ax.set_title(gene, fontstyle='italic', pad=15)
    ax.text(-0.18, 1.04, chr(65+panel), transform=ax.transAxes, fontweight='bold', fontsize=14)
fig2.subplots_adjust(left=0.12, right=0.96, bottom=0.28, top=0.87, wspace=0.35)
pd.DataFrame(detection_points).to_csv(OUT / 'Figure2_technical_points.csv', index=False, encoding='utf-8-sig')
save_figure(fig2, 'Figure2_mutant_nondetection')
plt.show()


<ipython-input-4-6a2c66773d65>:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 结果文字与图注

下面生成可用于论文初稿的英文描述与图注，并记录分析范围。现阶段使用 “increased” 描述估计倍数，不使用 “significantly increased”。

In [5]:
folds = wt_results.loc[wt_results.condition.eq('12hpe')].set_index('probe_name').fold_change
report = f"""# qPCR analysis — 2026-09-20

## Results draft
After normalization to Tub and calibration against baseline WT, AoGPR200 and AoGPRPR8 transcript abundance in WT at 12 hpe was {folds['AoGPR200']:.2f}-fold and {folds['AoGPRPR8']:.2f}-fold that at baseline, respectively. Neither receptor transcript was detected in its corresponding deletion mutant at baseline or 12 hpe (0/3 technical reactions for each gene-condition combination), whereas Tub was detected in all reactions. These observations are based on one cDNA sample per condition measured in technical triplicate; biological variability and statistical significance were not assessed.

## Figure 1. Receptor transcript induction in WT at 12 hpe.
Relative AoGPR200 and AoGPRPR8 transcript abundance was calculated using the 2^(-ΔΔCt) method, with Tub as the reference gene and baseline WT as the calibrator for each target. Bars represent estimates calculated from the mean Cp of three technical reactions per assay. Black dots denote individual target technical reactions normalized to the sample mean Tub Cp and the mean baseline WT delta Ct; they show target-assay technical variation and do not include reference or calibrator uncertainty. Bar heights are the geometric means of these transformed technical values, rather than their arithmetic means. Whiskers span ±1 propagated technical standard error in ΔΔCt space, transformed to the fold-change scale; they are not biological error bars or confidence intervals. Baseline calibrators equal 1 by definition. One cDNA sample was analyzed per condition; no significance tests were performed. Comparable, near-100% amplification efficiencies are assumed.

## Figure 2. Receptor transcripts were not detected in the corresponding deletion mutants.
The proportion of target-positive technical reactions is shown for AoGPR200 (A) and AoGPRPR8 (B) in WT and their corresponding deletion mutants at baseline and 12 hpe. Black dots denote individual reactions scored as detected (100) or not detected (0); zero within-group SD gives no visible error bars. Pale green denotes WT and purple denotes the deletion mutant. Delta200 and DeltaPR8 abbreviate the corresponding full mutant names. WT targets were detected in 3/3 reactions, whereas the corresponding mutant targets were not detected in any of the three reactions. ND indicates input Cp values coded as 0; zero-height bars represent 0% detection, not a quantified zero transcript abundance. Tub was detected in 3/3 reactions in every sample. The detection limit cannot be determined from the summary table alone.

## Plot style
The supplied manuscript style is used: #BEDEC7 controls, #7B61AA comparison groups, black outlines, bar width 0.7, black technical dots (alpha 0.7), 45-degree bold x labels, classic axes, no legend. Both figures are 7 x 5 inches, with two approximately 3.5 x 5 inch gene panels. Styling does not change the experimental unit or add ANOVA/Tukey tests.

## Reproducibility
Input: {SOURCE}
Local input snapshot: {SNAPSHOT.name}
Cp=0: not detected; never used as Ct=0 in normalization.
No replicate exclusions. Missing mutant expression fold changes remain blank.
"""
(OUT / 'Results_and_figure_legends.md').write_text(report, encoding='utf-8')
print(f'AoGPR200: {folds["AoGPR200"]:.3f}-fold; AoGPRPR8: {folds["AoGPRPR8"]:.3f}-fold')
print(f'图形、源数据和英文图注已保存至 {OUT}')


AoGPR200: 3.272-fold; AoGPRPR8: 4.438-fold
图形、源数据和英文图注已保存至 /Users/alvinl/Documents/ChatGPT/ijms文章case/final_polish_20260920/output/qRT_PCR_source_data/reproduced
